# 선택적 저조도 보정 + YOLO11n: 처음부터 실행하기

**사용 순서:** GitHub에 코드 업로드 → 이 노트북을 Colab에서 열기 → `파일 > Drive에 사본 저장` → `런타임 > 런타임 유형 변경 > GPU`.

처음에는 `MODE="smoke"`로 **3-2까지만** 진행하세요. 정상 동작을 확인하면 `MODE="full"`로 바꿔 **0단계부터 다시 실행**합니다. 셀은 위에서부터 하나씩 `Shift+Enter`로 실행합니다. `모두 실행`은 사용하지 마세요.

- 학습 모델: D=원본, E=항상 보정본, M=고정 50:50 혼합. 모델마다 학습 이미지 수와 epoch가 같습니다.
- M은 원본 이미지 한 장당 원본/보정본 중 한 버전만 사용합니다. D/E보다 학습량이 두 배가 되지 않습니다.
- 기준값은 검증셋에서 선택합니다. 최종 테스트를 시작하면 기준값을 다시 고르는 작업은 잠깁니다.
- 코드·노트북만 GitHub에 올립니다. 원자료·보정 이미지·정답 라벨·개별 예측·가중치는 GitHub 업로드용 결과 ZIP에 포함되지 않습니다.

**검증 범위:** 제작 시 실제 사전학습 모델의 CPU 추론과 소규모 가상 데이터 학습, 변환·평가·저장 코드의 동작을 점검했습니다. 실제 ExDark 전체 학습과 Colab GPU 성능은 여러분이 실행해 확인해야 합니다. 성능 숫자는 미리 채워 놓지 않았습니다.

## 0-1. 설정 — 처음에는 GitHub 아이디만 바꾸세요

`GITHUB_USER`에 저장소 주소의 아이디 부분을 넣으세요. 예: `https://github.com/honggil/lowlight-adaptive-yolo`이면 `honggil`입니다. 이메일 주소가 아닙니다.

처음에는 `MODE="smoke"`를 유지합니다. 정상 작동 확인 후 `"full"`로 바꾸세요. 본 실험의 D/E/M은 모두 같은 EPOCHS/BATCH/SEED로 실행합니다.

In [ ]:
GITHUB_USER = "YOUR_GITHUB_ID"  # 이 부분만 본인 GitHub 아이디로 바꾸세요.
REPO_NAME = "lowlight-adaptive-yolo"
MODE = "smoke"                # 첫 실행: smoke / 본 실험: full
RUN_TAG = "v1"                # 중단 후 이어하기는 같은 이름 유지
EPOCHS = 30                    # 본 실험 D/E/M 모두 동일
BATCH = 16                     # GPU 메모리 부족이면 8로 바꿔 새 실행으로 진행
SEED = 0

# 아래 두 경로는 그대로 두세요.
DRIVE_FOLDER = "lowlight_lab"
PROJECT_FOLDER = "/content/lowlight-adaptive-yolo"


## 0-2. Google Drive 연결 + GitHub 코드 받기

Google 계정 선택과 Drive 접근 허용 창이 나오면 본인 계정으로 진행합니다. 공개 저장소를 복제하므로 GitHub 비밀번호나 개인 토큰은 넣지 않습니다.

`Repository not found`가 나오면 저장소 이름, 아이디, 공개 여부를 확인하세요. 저장소 첫 화면에 `src`, `notebooks`, `requirements.txt`가 바로 있어야 합니다.

In [ ]:
from pathlib import Path
import os, re, subprocess, sys
from google.colab import drive

if GITHUB_USER == "YOUR_GITHUB_ID" or not re.fullmatch(r"[A-Za-z0-9-]+", GITHUB_USER):
    raise ValueError("0-1의 GITHUB_USER를 실제 GitHub 아이디로 바꾸고 다시 실행하세요.")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", REPO_NAME):
    raise ValueError("저장소 이름을 확인하세요.")
drive.mount("/content/drive")
PROJECT = Path(PROJECT_FOLDER)
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True, env=env)
else:
    current = subprocess.check_output(["git", "-C", str(PROJECT), "remote", "get-url", "origin"], text=True).strip()
    if current.rstrip("/").removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError("이미 다른 저장소가 있습니다. 런타임 연결 해제 및 삭제 후 다시 시작하세요.")
    print("기존 코드 재사용: 이번 실행 중에 코드를 자동으로 바꾸지 않습니다.")
if not (PROJECT / "src" / "lab.py").is_file():
    raise RuntimeError("저장소 폴더 구조가 다릅니다. 압축을 푼 폴더 안의 내용물을 GitHub 최상위에 올렸는지 확인하세요.")
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print("프로젝트:", PROJECT)
print("버전:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
STORAGE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
STORAGE.mkdir(parents=True, exist_ok=True)


## 0-3. 패키지 설치

Colab의 GPU용 PyTorch는 그대로 활용하고, 실험에 필요한 패키지를 설치합니다. 설치 완료 후 다음 셀로 넘어가세요. 런타임을 재시작하라는 안내가 실제로 표시되면 재시작 후 0-1부터 다시 실행합니다.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")], check=True)
print("패키지 설치 완료")


## 0-4. GPU·모델 준비

GPU 이름이 출력되면 정상입니다. T4가 흔하지만 배정되는 종류는 달라질 수 있습니다. GPU가 없다고 나오면 런타임 유형을 바꾸세요.

모델 가중치는 Drive에 저장합니다. 공식 Zero-DCE++ 코드 버전과 가중치 해시, Python·라이브러리 버전도 기록합니다.

In [ ]:
from src.lab import Lab
from IPython.display import display, Markdown

lab = Lab(storage_root=STORAGE, mode=MODE, tag=RUN_TAG,
          epochs=EPOCHS, batch=BATCH, seed=SEED, imgsz=640, max_side=640)
environment = lab.setup()
print("준비 완료:", lab.run_id)


## 1-1. ExDark 다운로드와 변환

첫 실행은 원본 이미지 ZIP 전체를 받습니다. `smoke`도 다운로드 크기는 같고, 실제 학습에 쓰는 이미지 수만 줄어듭니다. 네트워크 상태에 따라 시간이 걸립니다.

정상적인 공식 분할은 train=3000, val=1800, test=2563입니다. 누락된 파일을 임의로 제외하지 않고 원인을 알려 주도록 만들었습니다. 숫자가 다른 상태에서 학습을 계속하지 마세요.

다운로드 제한이 생기면 아래 두 파일을 브라우저에서 받아 Drive의 **내 드라이브/lowlight_lab/raw/exdark/**에 지정 이름으로 넣고 이 셀을 다시 실행하세요.

- [이미지 ZIP](https://drive.google.com/file/d/1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx/view) → `ExDark.zip`
- [라벨 ZIP](https://drive.google.com/file/d/1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i/view) → `ExDark_Annno.zip`

출처: [ExDark 공식 저장소](https://github.com/cs-chan/Exclusively-Dark-Image-Dataset). 원자료 사용 조건을 확인하고 학술 실험에 사용하세요.

In [ ]:
counts = lab.prepare_data()
print(counts)


## 1-2. 정답 박스를 눈으로 확인

초록 박스가 물체를 감싸는지, 사람·자동차 등의 이름이 맞는지 확인하세요. 박스가 전부 한쪽으로 치우치거나 크기가 이상하면 다음 단계로 진행하지 말고 해당 화면과 오류를 확인합니다.

In [ ]:
lab.show_labels(n=6)


## 2. D/E/M 입력 만들기

D는 원본, E는 Zero-DCE++ 보정본, M은 고정된 50:50 혼합입니다. 모든 경로가 동일하게 긴 변 최대 640픽셀로 줄이고, 박스 비율을 유지합니다. 보정 모델의 내부 연산은 12배 축소 설정이며, 가장자리는 패딩 후 되돌려 박스가 잘리지 않게 합니다.

PNG는 현재 Colab 디스크에 만들어 학습 속도를 확보합니다. 세션이 삭제되면 이 단계에서 다시 만듭니다. Drive에는 원자료 ZIP과 모델·실험 결과가 남습니다.

In [ ]:
lab.prepare_views()


## 3-1. 먼저 D를 학습해 작동 확인

`MODE="smoke"`이면 학습 최대 120장·검증 최대 60장으로 2 epoch만 실행합니다. `MODE="full"`이면 공식 전체 학습·검증 데이터와 EPOCHS 설정을 사용합니다.

중단 후 같은 설정으로 다시 실행하면 `last.pt`에서 이어갑니다. 이미 완료한 경우 다시 학습하지 않습니다.

In [ ]:
best_d = lab.train("D")
print("D 완료:", best_d)


## 3-2. 첫 결과 화면 확인 — 처음에는 여기까지

정답을 아는 검증 이미지 한 장에서 원본/항상 보정/선택 보정 입력을 나란히 보여 줍니다. 아직 기준값은 임시값 60이고, 모델은 D입니다. 이는 작동 확인용이며 제안 방법의 최종 성능을 의미하지 않습니다.

**2 epoch가 끝나고 화면이 나오면:** 맨 위에서 `MODE="full"`로 변경 → **0-1부터 3-1까지 다시 실행** → 아래 4단계부터 진행하세요. `smoke` 결과는 본 실험과 다른 폴더에 저장됩니다.

In [ ]:
preview_record = next(r for r in lab.exdark if r["split"] == "val")
lab.demo(preview_record["path"], model_name="D", threshold=60)


## 4-1. E 학습 — 본 실험

보정본으로 학습하는 모델입니다. D와 같은 epoch, 배치, seed를 사용합니다. 하루에 모두 끝낼 필요는 없습니다. 다음 접속 때 0단계, 1단계, 2단계를 복구한 뒤 이 셀을 실행하면 됩니다.

In [ ]:
if MODE != "full":
    print("먼저 MODE='full'로 바꾸고 0단계부터 다시 실행하세요.")
else:
    best_e = lab.train("E")
    print("E 완료:", best_e)


## 4-2. M 학습 — 본 실험

원본·보정본 혼합 모델입니다. 학습 이미지 수는 D/E와 같습니다. M의 체크포인트는 미리 고정한 50:50 검증 입력에서 선택합니다. 이후 동일한 M 가중치로 입력 정책만 바꿔 비교합니다.

In [ ]:
if MODE == "full":
    best_m = lab.train("M")
    print("M 완료:", best_m)


## 5. A·B·C 기초 비교 — 검증 데이터

사전학습 YOLO에서 원본(A), 감마/CLAHE(B), Zero-DCE++(C)를 비교합니다. 빠른 기초 확인을 위해 검증 데이터 중 고정 표본을 씁니다. 최종 결과는 8단계에서 동일한 테스트셋으로 다시 계산합니다.

이 단계의 숫자는 `baseline_validation.csv`에 저장되므로 최종 테스트 표와 구분합니다.

In [ ]:
if MODE == "full":
    display(lab.baselines())


## 6. 밝은 이미지 준비

COCO **val2017**에서 원본 평균 밝기 90 이상이며 공통 클래스의 정답 박스가 있는 이미지를 고릅니다. 미리 고정한 seed로 검증용 300장·테스트용 500장을 서로 겹치지 않게 뽑습니다. COCO train 이미지를 테스트에 사용하지 않습니다.

촬영 시간이 검증되지 않았으므로 '낮/밤' 대신 '밝은 이미지/저조도 이미지'라고 부릅니다. COCO 첫 다운로드도 시간이 걸리며 ZIP은 Drive에 백업됩니다.

In [ ]:
if MODE == "full":
    lab.prepare_bright()


## 7. 검증셋에서 기준값 선택·고정

ExDark 검증 300장 + COCO 밝은 검증 300장으로 기준값 0,30,60,90,120,256을 비교합니다.

- 0: 보정 없음 / 256: 항상 보정 / 중간값: 선택적 보정.
- M-항상 대비 전체·저조도·밝은 그룹의 mAP50 감소가 각각 **1%p 이내**인 후보 중 측정 처리 시간이 가장 짧은 값을 선택합니다.
- 1%p는 이 프로젝트가 사전에 정한 허용 손실입니다. 결과를 본 뒤 바꾸지 않습니다.
- 0이나 256이 최종 선택되어도 그대로 보고하세요. 선택적 보정의 이득을 미리 가정하지 않습니다.

선택 결과는 `protocol.json`에 저장됩니다. 최종 테스트를 시작하면 이 단계의 재실행은 차단됩니다.

In [ ]:
if MODE == "full":
    selected_protocol = lab.tune()
    print("선택된 기준값:", selected_protocol["selected_threshold"])


## 8. 최종 테스트

학습에 쓰지 않은 ExDark 테스트 2,563장과 밝은 COCO 테스트 500장으로 예측합니다. ExDark 전체 결과와 별도의 500+500 혼합 결과를 계산합니다. 혼합 mAP는 두 그룹 평균이 아니라 실제 예측을 합쳐 계산합니다.

`include_pretrained=True`는 A·B·C까지 최종 표에 포함합니다. 중간에 끊기면 같은 셀을 다시 실행하세요. 완성된 방법별 예측 파일은 재사용합니다.

최종 mAP는 공식 COCO 평가기(maxDets=100)로 계산합니다. 학습 화면의 Ultralytics 지표와 완전히 같은 계산은 아니므로 최종 비교는 이 표끼리 하세요.

In [ ]:
if MODE == "full":
    display(lab.final_test(include_pretrained=True))


## 9-1. 실제 처리 시간 측정

동일한 혼합 테스트 표본 최대 300장을 세 번 반복합니다. 예열 후 한 장씩 처리하며, 밝기 판단·실제 보정·리사이즈·YOLO 전처리·전송·추론·NMS를 포함합니다. 파일 읽기·이미지 디코딩·모델 로딩은 제외합니다.

다섯 방법을 같은 GPU 세션에서 이어서 실행합니다. `mean_ms`는 낮을수록, `fps`는 높을수록 좋습니다. 보정본 캐시만 읽어 속도를 재지 않습니다.

In [ ]:
if MODE == "full":
    display(lab.benchmark())


## 9-2. 표·그래프 자동 생성

실측값으로 요약표와 네 개의 그래프를 만듭니다. 개선이 없는 경우에도 실제 차이를 그대로 적습니다.

In [ ]:
if MODE == "full":
    report_files = lab.report()
    display(Markdown(Path(report_files["summary"]).read_text(encoding="utf-8")))
    from IPython.display import Image as DisplayImage
    for figure_path in report_files["figures"]:
        display(DisplayImage(filename=figure_path))


## 10. 내 사진으로 시연 — 선택 사항

파일 선택 창에서 사진 한 장을 올립니다. 출력 화면은 Colab 안에 표시됩니다. 웹 서버나 별도 공개 링크가 필요하지 않습니다. 모델은 M, 기준값은 7단계에서 고정한 값을 씁니다.

시연용 박스 표시 confidence=0.25, 정량 AP 계산 confidence=0.001입니다. 시연 화면의 박스 개수만으로 성능을 판단하지 않습니다.

In [ ]:
if MODE == "full":
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        selected_file = next(iter(uploaded))
        lab.demo(selected_file, model_name="M")


## 11. GitHub에 올릴 결과 다운로드

아래 셀은 집계 CSV·그래프·환경·설정·학습 로그만 묶습니다. 다운로드한 `results_for_github.zip`을 PC에서 압축 해제한 뒤, 안의 `results` 폴더를 GitHub 저장소에 업로드하세요. ZIP 자체를 올리지 않습니다.

학습된 best.pt는 Drive의 실행 폴더에 남습니다. 과제에서 가중치 제출이 필요하면 D/E/M 파일을 구분해 별도로 제출하세요. 모든 원본 이미지와 라벨을 공개 저장소에 넣지 마세요.

In [ ]:
if MODE == "full":
    from google.colab import files
    export_path = lab.export()
    files.download(export_path)


## 다음 날 다시 시작할 때

1. 같은 Drive 사본을 엽니다. GPU를 연결합니다.
2. 0-1~0-4를 실행합니다. 이전과 같은 MODE/RUN_TAG/EPOCHS/BATCH/SEED를 유지합니다.
3. 1-1 → 2를 실행해 원자료·학습 입력을 복구합니다.
4. 중단된 D/E/M 학습 셀을 실행합니다. `last.pt`가 있으면 자동으로 이어갑니다.
5. 학습이 끝났고 기준값까지 고정했다면 6단계로 COCO를 복구하고 **7단계를 건너뛰어** 8단계 또는 9단계를 이어갑니다.

### 자주 생기는 문제

- **GPU 없음:** 런타임 유형을 GPU로 바꿉니다. 무료 GPU 배정은 보장되지 않습니다.
- **CUDA out of memory:** BATCH=8로 바꾸고 0단계부터 새 실행으로 진행합니다. 새 배치 설정의 D/E/M을 모두 맞춥니다.
- **저장소를 찾지 못함:** GitHub 아이디·저장소 이름·공개 여부를 확인합니다.
- **ZIP 다운로드 제한:** 1단계의 수동 다운로드 안내를 따릅니다.
- **원자료·라벨 누락:** 오류에 표시된 파일을 확인합니다. 이미지 수가 줄어든 채 진행하지 않습니다.
- **코드 수정 반영:** GitHub에 수정 후 새 런타임에서 다시 코드를 받습니다. 실험 도중 코드 변경은 새 RUN_TAG로 구분하고 기록합니다.
- **테스트 이후 기준값 재선택 오류:** 정상적인 보호 동작입니다. 저장된 protocol.json을 사용해 8단계를 이어갑니다.

공식 안내: [GitHub 파일 업로드](https://docs.github.com/en/repositories/working-with-files/managing-files/adding-a-file-to-a-repository), [Colab FAQ](https://research.google.com/colaboratory/faq.html), [YOLO11](https://docs.ultralytics.com/models/yolo11/), [Zero-DCE++](https://github.com/Li-Chongyi/Zero-DCE_extension).